# Ticha HTR — evaluation of the fine-tuned TrOCR models

Evaluates the models trained in `ticha_trocr_finetuning.ipynb` on the held-out data and gathers the results and charts:

- **Models:** the pretrained `microsoft/trocr-base-handwritten` (zero-shot) and the three fine-tuned runs saved in `~/ticha-runs/` (5 epochs; 12 epochs with label smoothing and cosine schedule; the same with data augmentation). Each run folder holds its best checkpoint (lowest validation CER).
- **Data:** `original/test` (real scans) and `augmented/test_degraded` (one degraded copy of each test line), cleaned with the same rules as in training.
- **Outputs:** predictions, metrics and figures in `~/ticha-runs/evaluation/` (or `$TICHA_RUNS_DIR/evaluation`).

These are the three runs of the fine-tuning notebook only. The report's numbers, including run E of the ablation notebook and the final system (run E + 10 beams + right-to-left rerank), are in [`ticha_report_metrics.ipynb`](ticha_report_metrics.ipynb).

Run it on the GCP VM with the **Python (ticha-trocr)** kernel. Predictions are cached, so re-running only redraws tables and charts. Set `TICHA_EVAL_LIMIT=50` to try the notebook on 50 lines per split.

In [ ]:
import json
import os
import re

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
import jiwer
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from IPython.display import display
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

RUNS_DIR = os.path.expanduser(os.environ.get('TICHA_RUNS_DIR', '~/ticha-runs'))
LIMIT = int(os.environ.get('TICHA_EVAL_LIMIT', 0))  # 0 = all lines
EVAL_DIR = os.path.join(RUNS_DIR, 'evaluation')
if LIMIT:  # keep trial runs apart from the real results
    EVAL_DIR = os.path.join(EVAL_DIR, f'trial-limit{LIMIT}')
FIG_DIR = os.path.join(EVAL_DIR, 'figures')
PRED_DIR = os.path.join(EVAL_DIR, 'predictions')
for d in (FIG_DIR, PRED_DIR):
    os.makedirs(d, exist_ok=True)
print('Results go to', EVAL_DIR)


def show(df, pct_cols=(), signed=False):
    """Display a table with the given columns as percentages."""
    out = df.copy()
    for c in pct_cols:
        out[c] = out[c].map(lambda v: '' if pd.isna(v) else (f'{v:+.0%}' if signed else f'{v:.1%}'))
    display(out)

BASE_ID = 'microsoft/trocr-base-handwritten'
FT = os.path.join(RUNS_DIR, 'trocr-base-handwritten-finetuned')
# name -> (checkpoint, training run folder or None)
MODELS = {
    'Pretrained (zero-shot)': (BASE_ID, None),
    'Fine-tuned, 5 epochs': (FT, FT),
    'Smoothing + cosine, 12 epochs': (f'{FT}-smooth-cosine', f'{FT}-smooth-cosine'),
    'Same + augmentation': (f'{FT}-smooth-cosine-aug', f'{FT}-smooth-cosine-aug'),
}
FINETUNED = [m for m in MODELS if MODELS[m][1]]

# Generation settings used during training (the training section of the fine-tuning notebook)
MAX_TARGET_LENGTH, NUM_BEAMS, BATCH_SIZE = 64, 2, 16

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU (slow)')
for name, (path, _) in MODELS.items():
    ok = path == BASE_ID or os.path.exists(os.path.join(path, 'model.safetensors'))
    print(f'{"ok " if ok else "MISSING"}  {name}: {path}')

## Test data

The same cleaning as in training, with uncertain readings dropped as in validation and test: abbreviation expansions and glosses are deleted, and lines with damage, illegible text or editorial notes are dropped. The degraded split is restricted to copies of the test lines that survive the cleaning, so both splits cover the same lines.

**Dataset v1.1.** Ticha lists one manuscript twice: Za687 and Th687 are the same pages with the same transcription. v1.0 had Za687 in test and Th687 in train, so the fine-tuned models had seen it (they read it at 1–7% CER, against 80% for the pretrained model). v1.1 keeps train as it was and leaves Za687 out of test, which raises each model's test CER by about 0.4–0.5 points. Cached predictions are filtered to the loaded lines, so nothing is predicted again.

In [ ]:
# Same rules as the 'Clean transcriber notes' cell of the fine-tuning notebook (keep_guesses=False).
BRACKET_RE = re.compile(r'\[([^\[\]]*)\]|\(([^()]*)\)')
NOTE_RE = re.compile(
    r'[?…]|^\.+$|\.\.|\s|,|^x+$|^word$'
    r'|cortado|roto|torn|mancha|borrado|firmas|ilegible',
    re.IGNORECASE)


def clean_text(text):
    """Return the diplomatic text, or None when the line should be dropped."""
    text = str(text or '')
    notes = [square or round for square, round in BRACKET_RE.findall(text)]
    if any(NOTE_RE.search(note.strip()) for note in notes if not note.strip().startswith('=')):
        return None
    text = BRACKET_RE.sub('', text)
    if re.search(r'[\[\]()]', text) or '#' in text:
        return None
    return re.sub(r'\s+', ' ', text).strip() or None


def clean(ds):
    ds = ds.map(lambda texts: {'text_raw': texts, 'text': [clean_text(t) for t in texts]},
                input_columns='text', batched=True)
    return ds.filter(lambda t: t is not None, input_columns='text')


# v1.1: train as in v1.0; Za687 (the same pages as Th687 in train) is not in test.
REVISION = 'v1.1'
test = clean(load_dataset('maclenn77/ticha-htr-lines', 'original', split='test', revision=REVISION))
validation = clean(load_dataset('maclenn77/ticha-htr-lines', 'original', split='validation', revision=REVISION))
degraded = clean(load_dataset('maclenn77/ticha-htr-lines', 'augmented', split='test_degraded', revision=REVISION))
kept = set(test['id'])
degraded = degraded.filter(lambda s: s in kept, input_columns='source_id')
if LIMIT:
    test = test.select(range(min(LIMIT, len(test))))
    validation = validation.select(range(min(LIMIT, len(validation))))
    kept = set(test['id'])
    degraded = degraded.filter(lambda s: s in kept, input_columns='source_id')

SPLITS = {'test': test, 'test_degraded': degraded}
for split, ds in {**SPLITS, 'validation': validation}.items():
    print(f'{split}: {len(ds):,} lines, {len(set(ds["doc_id"]))} documents')

## Predictions

Each model reads every line with the generation settings used during training (beam search with 2 beams, at most 64 tokens). Predictions are saved per model and split, and loaded from disk on later runs.

In [ ]:
def slug(name):
    return re.sub(r'[^a-z0-9]+', '-', name.lower()).strip('-')


def predict(path, ds):
    processor = TrOCRProcessor.from_pretrained(path if path != BASE_ID else BASE_ID)
    model = VisionEncoderDecoderModel.from_pretrained(path).to(device).eval()
    if path == BASE_ID:  # the saved runs carry these already
        model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
        model.config.pad_token_id = processor.tokenizer.pad_token_id
        model.config.eos_token_id = processor.tokenizer.sep_token_id
    use_bf16 = device == 'cuda' and torch.cuda.is_bf16_supported()
    preds = []
    for start in range(0, len(ds), BATCH_SIZE):
        batch = ds[start:start + BATCH_SIZE]
        pixel_values = processor(images=[im.convert('RGB') for im in batch['image']],
                                 return_tensors='pt').pixel_values.to(device)
        with torch.no_grad(), torch.autocast(device, dtype=torch.bfloat16, enabled=use_bf16):
            ids = model.generate(pixel_values, num_beams=NUM_BEAMS, max_length=MAX_TARGET_LENGTH,
                                 early_stopping=True)
        preds += [p.strip() for p in processor.batch_decode(ids, skip_special_tokens=True)]
    del model
    torch.cuda.empty_cache()
    return preds


def line_errors(refs, hyps):
    """Character edits and reference length of each line, so CER can be summed over any subset."""
    edits, lengths = [], []
    for r, h in zip(refs, hyps):
        out = jiwer.process_characters(r, h)
        edits.append(out.substitutions + out.deletions + out.insertions)
        lengths.append(out.substitutions + out.deletions + out.hits)
    return edits, lengths


frames = []
for name, (path, _) in MODELS.items():
    for split, ds in SPLITS.items():
        cache = os.path.join(PRED_DIR, f'{slug(name)}__{split}.csv')
        if os.path.exists(cache):
            df = pd.read_csv(cache, keep_default_na=False)
            df = df[df['id'].isin(set(ds['id']))].reset_index(drop=True)  # the cache may hold lines left out since
        else:
            print(f'Predicting: {name} on {split} ({len(ds):,} lines)...')
            df = pd.DataFrame({'id': ds['id'], 'doc_id': ds['doc_id'], 'language': ds['language'],
                               'reference': ds['text'], 'prediction': predict(path, ds)})
            df.to_csv(cache, index=False)
        df['edits'], df['ref_chars'] = line_errors(df['reference'], df['prediction'])
        df['line_cer'] = df['edits'] / df['ref_chars']
        frames.append(df.assign(model=name, split=split))

preds = pd.concat(frames, ignore_index=True)
preds['model'] = pd.Categorical(preds['model'], categories=list(MODELS), ordered=True)
print(f'{len(preds):,} predictions')

# Zero-shot baseline on validation: the reference line of the validation-CER chart
cache = os.path.join(PRED_DIR, 'pretrained-zero-shot__validation.csv')
if os.path.exists(cache):
    base_val = pd.read_csv(cache, keep_default_na=False)
    base_val = base_val[base_val['id'].isin(set(validation['id']))].reset_index(drop=True)
else:
    print(f'Predicting: pretrained on validation ({len(validation):,} lines)...')
    base_val = pd.DataFrame({'id': validation['id'], 'reference': validation['text'],
                             'prediction': predict(BASE_ID, validation)})
    base_val.to_csv(cache, index=False)
e, n = line_errors(base_val['reference'], base_val['prediction'])
BASELINE_VAL_CER = sum(e) / sum(n)
print(f'Pretrained baseline, validation CER: {BASELINE_VAL_CER:.1%}')

## Overall results

CER is the share of reference characters that must be inserted, deleted or replaced to turn the prediction into the reference; WER is the same for words. Both are computed over all lines at once (total edits / total reference length), like `jiwer`. The 95% interval comes from resampling the test lines 2,000 times (bootstrap), so it reflects how much the score depends on which lines are in the test set, not the variation between training runs.

In [ ]:
rng = np.random.default_rng(0)
N_BOOT = 2000


def bootstrap(edits, lengths, idx):
    return edits[idx].sum(axis=1) / lengths[idx].sum(axis=1)


rows, boot = [], {}
for (model, split), g in preds.groupby(['model', 'split'], observed=True):
    e, n = g['edits'].to_numpy(), g['ref_chars'].to_numpy()
    idx = rng.integers(0, len(g), size=(N_BOOT, len(g)))
    boot[model, split] = bootstrap(e, n, idx)
    lo, hi = np.percentile(boot[model, split], [2.5, 97.5])
    rows.append({'model': model, 'split': split, 'lines': len(g),
                 'CER': e.sum() / n.sum(), 'CER 95% low': lo, 'CER 95% high': hi,
                 'WER': jiwer.wer(list(g['reference']), list(g['prediction'])),
                 'exact lines': (g['edits'] == 0).mean()})
summary = pd.DataFrame(rows)
summary.to_csv(os.path.join(EVAL_DIR, 'summary.csv'), index=False)

show(summary, ['CER', 'CER 95% low', 'CER 95% high', 'WER', 'exact lines'])

In [ ]:
# Is the best of these three runs better than the first one? Paired bootstrap on the same resampled lines.
best, first = 'Same + augmentation', 'Fine-tuned, 5 epochs'
for split in SPLITS:
    a = preds[(preds.model == best) & (preds.split == split)].set_index('id')
    b = preds[(preds.model == first) & (preds.split == split)].set_index('id').loc[a.index]
    idx = rng.integers(0, len(a), size=(N_BOOT, len(a)))
    diff = (bootstrap(a['edits'].to_numpy(), a['ref_chars'].to_numpy(), idx)
            - bootstrap(b['edits'].to_numpy(), b['ref_chars'].to_numpy(), idx))
    lo, hi = np.percentile(diff, [2.5, 97.5])
    print(f'{split}: CER difference {diff.mean():+.1%} (95% interval {lo:+.1%} to {hi:+.1%}); '
          f'best model better in {(diff < 0).mean():.0%} of resamples')

## Charts

Colors follow the run in every chart: blue for the 5-epoch run, orange for label smoothing + cosine, aqua for the same with augmentation. The dotted gray line is the pretrained model before fine-tuning (the zero-shot baseline). Figures are saved as PNG in `evaluation/figures/`.

In [ ]:
COLORS = {
    'Pretrained (zero-shot)': '#8a8984',
    'Fine-tuned, 5 epochs': '#2a78d6',
    'Smoothing + cosine, 12 epochs': '#eb6834',
    'Same + augmentation': '#1baf7a',
}
INK, MUTED, GRID = '#0b0b0b', '#52514e', '#e4e3df'
BASE = 'Pretrained (zero-shot)'
BASELINE_STYLE = dict(color=COLORS[BASE], linestyle=':', linewidth=2)
plt.rcParams.update({
    'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'savefig.facecolor': '#fcfcfb',
    'axes.edgecolor': GRID, 'axes.labelcolor': MUTED, 'axes.titlecolor': INK,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8,
    'xtick.color': MUTED, 'ytick.color': MUTED, 'legend.frameon': False, 'font.size': 10,
    'lines.linewidth': 2,
})
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, PercentFormatter


def save(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, f'{name}.png'), dpi=150, bbox_inches='tight')
    plt.show()


def history(run_dir):
    with open(os.path.join(run_dir, 'trainer_state.json')) as f:
        return json.load(f)['log_history']

### Training: validation CER per epoch

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ends = []
for name in FINETUNED:
    ev = [h for h in history(MODELS[name][1]) if 'eval_cer' in h]
    x, y = [h['epoch'] for h in ev], [h['eval_cer'] for h in ev]
    ax.plot(x, y, color=COLORS[name], marker='o', markersize=5, label=name)
    ends.append((x[-1], y[-1]))
for x, y in ends:  # end labels; nudge apart when two runs end at the same epoch with close values
    near = [yy for xx, yy in ends if xx == x and yy != y and abs(yy - y) < 0.02]
    dy = 0 if not near else (5 if y > near[0] else -5)
    ax.annotate(f'{y:.1%}', (x, y), xytext=(6, dy), textcoords='offset points', va='center', color=INK, fontsize=9)
ax.axhline(BASELINE_VAL_CER, label=f'Pretrained baseline ({BASELINE_VAL_CER:.1%})', **BASELINE_STYLE)
ax.set(title='Validation CER per epoch', xlabel='Epoch', ylabel='Validation CER', ylim=(0, None))
ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.set_xticks(range(1, 13))
ax.legend(loc='center right')
save(fig, '01_validation_cer_per_epoch')

### Training: learning-rate schedules

The linear schedule of the first run lowers the rate by the same amount every step; the cosine schedule keeps it high for longer and eases it to zero at the end. Both start with a warm-up over the first 5% of the steps.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name in ['Fine-tuned, 5 epochs', 'Smoothing + cosine, 12 epochs']:
    lr = [h for h in history(MODELS[name][1]) if 'learning_rate' in h]
    total = max(h['epoch'] for h in history(MODELS[name][1]))
    x = [h['epoch'] / total for h in lr]
    ax.plot(x, [h['learning_rate'] for h in lr], color=COLORS[name],
            label='Linear (5-epoch run)' if '5 epochs' in name else 'Cosine (both 12-epoch runs)')
ax.set(title='Learning rate during training', xlabel='Share of training completed', ylabel='Learning rate')
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: '0' if v == 0 else f'{v:.0e}'.replace('e-0', 'e-')))
ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.legend(loc='upper right')
save(fig, '02_learning_rate_schedules')

### Training: loss of the first run

Training loss keeps falling while validation loss stops improving after epoch 4: the model starts to memorize the training lines. (The runs with label smoothing are not shown: smoothing puts a floor of about 1.41 under their loss, so their losses are not comparable with this one.)

In [ ]:
name = 'Fine-tuned, 5 epochs'
hist = history(MODELS[name][1])
train = [h for h in hist if 'loss' in h]
ev = [h for h in hist if 'eval_loss' in h]
fig, ax = plt.subplots(figsize=(8, 4))
c = COLORS[name]
ax.plot([h['epoch'] for h in train], [h['loss'] for h in train], color=c, label='Training loss')
ax.plot([h['epoch'] for h in ev], [h['eval_loss'] for h in ev], color=c, linestyle='--', marker='o',
        markersize=5, label='Validation loss')
for series, key, text in [(train, 'loss', 'training'), (ev, 'eval_loss', 'validation')]:
    ax.annotate(text, (series[-1]['epoch'], series[-1][key]), xytext=(6, 0),
                textcoords='offset points', va='center', color=INK, fontsize=9)
ax.set(title=f'Loss per epoch: {name}', xlabel='Epoch', ylabel='Cross-entropy loss', ylim=(0, None))
ax.legend(loc='upper right')
save(fig, '03_loss_first_run')

### Test results: real and degraded scans

Dots are the CER on each split; bars are the 95% bootstrap intervals. The dotted lines mark the pretrained model on each split.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
for i, name in enumerate(FINETUNED):
    for split, offset in [('test', -0.12), ('test_degraded', 0.12)]:
        r = summary[(summary.model == name) & (summary.split == split)].iloc[0]
        filled = split == 'test'
        ax.errorbar(r['CER'], i + offset, xerr=[[r['CER'] - r['CER 95% low']], [r['CER 95% high'] - r['CER']]],
                    fmt='o', markersize=8, color=COLORS[name], markerfacecolor=COLORS[name] if filled else '#fcfcfb',
                    markeredgewidth=2, elinewidth=1.5, capsize=0)
        ax.annotate(f'{r["CER"]:.1%}', (r['CER 95% high'], i + offset), xytext=(6, 0), textcoords='offset points',
                    va='center', color=INK, fontsize=9)
for split, label in [('test', 'real'), ('test_degraded', 'degraded')]:
    v = summary[(summary.model == BASE) & (summary.split == split)]['CER'].iloc[0]
    ax.axvline(v, **BASELINE_STYLE)
    ax.annotate(f'{label} {v:.1%}', (v, 1), xycoords=('data', 'axes fraction'), xytext=(0, 3),
                textcoords='offset points', ha='right' if split == 'test' else 'left', va='bottom',
                color=MUTED, fontsize=8)
ax.set_xlim(0, None)
ax.set_yticks(range(len(FINETUNED)), FINETUNED)
ax.invert_yaxis()
ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.set_xlabel('CER (lower is better)')
ax.set_title('Test CER of the fine-tuned models', pad=16)
ax.grid(axis='y', visible=False)
ax.legend(handles=[Line2D([], [], marker='o', linestyle='', color=MUTED, markersize=8, label='Real scans (test)'),
                   Line2D([], [], marker='o', linestyle='', color=MUTED, markerfacecolor='#fcfcfb',
                          markeredgewidth=2, markersize=8, label='Degraded scans (test_degraded)'),
                   Line2D([], [], label='Pretrained baseline', **BASELINE_STYLE)],
          loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=3)
save(fig, '04_test_cer_real_and_degraded')

### Test results by language

In [ ]:
by_lang = (preds.groupby(['split', 'language', 'model'], observed=True)[['edits', 'ref_chars']].sum()
           .assign(CER=lambda d: d['edits'] / d['ref_chars']).reset_index())
lines_per_lang = preds[(preds.split == 'test') & (preds.model == FINETUNED[0])]['language'].value_counts()
by_lang.pivot_table(index=['split', 'language'], columns='model', values='CER', observed=True).to_csv(
    os.path.join(EVAL_DIR, 'cer_by_language.csv'))

langs = list(lines_per_lang.index)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
height = 0.8 / len(FINETUNED)
for ax, split in zip(axes, SPLITS):
    for j, name in enumerate(FINETUNED):
        d = by_lang[(by_lang.split == split) & (by_lang.model == name)].set_index('language').reindex(langs)
        y = np.arange(len(langs)) + (j - 1) * height
        ax.barh(y, d['CER'], height=height - 0.03, color=COLORS[name], label=name)
        for yy, v in zip(y, d['CER']):
            ax.annotate(f'{v:.1%}', (v, yy), xytext=(4, 0), textcoords='offset points', va='center',
                        color=INK, fontsize=8)
    base = by_lang[(by_lang.split == split) & (by_lang.model == BASE)].set_index('language').reindex(langs)
    for k, v in enumerate(base['CER']):
        ax.vlines(v, k - 0.45, k + 0.45, **BASELINE_STYLE)
        ax.annotate(f'{v:.0%}', (v, k - 0.45), xytext=(3, 2), textcoords='offset points', color=MUTED, fontsize=8)
    ax.set_yticks(range(len(langs)), [f'{l}\n({lines_per_lang[l]} lines)' for l in langs])
    ax.set_title('Real scans (test)' if split == 'test' else 'Degraded scans (test_degraded)')
    ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=0))
    ax.set_xlabel('CER (lower is better)')
    ax.grid(axis='y', visible=False)
axes[0].invert_yaxis()
handles, labels = axes[0].get_legend_handles_labels()
handles.append(Line2D([], [], **BASELINE_STYLE)); labels.append('Pretrained baseline')
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=len(FINETUNED) + 1)
fig.suptitle('Test CER by language', x=0.01, ha='left', fontweight='bold')
save(fig, '05_test_cer_by_language')

### How errors are spread over lines

Share of test lines (y) whose CER is at most x. A curve further up and to the left is better; the height at x = 0 is the share of lines read without any error.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for name in FINETUNED + [BASE]:
    v = np.sort(preds[(preds.model == name) & (preds.split == 'test')]['line_cer'].clip(upper=1).to_numpy())
    y = np.arange(1, len(v) + 1) / len(v)
    style = BASELINE_STYLE if name == BASE else {'color': COLORS[name]}
    label = 'Pretrained baseline' if name == BASE else name
    ax.step(v, y, where='post', label=f'{label} ({(v == 0).mean():.0%} exact)', **style)
ax.set(title='Distribution of line CER on the test set', xlabel='Line CER (clipped at 100%)',
       ylabel='Share of lines', xlim=(0, 1), ylim=(0, 1))
ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.legend(loc='lower right')
save(fig, '06_line_cer_distribution')

### Test results by document

Each row is a test document: the 5-epoch run and the augmented run, joined by a line, and the pretrained baseline as a dotted mark. Documents are sorted by the augmented model's CER.

In [ ]:
by_doc = (preds[preds.split == 'test'].groupby(['doc_id', 'model'], observed=True)
          .agg(edits=('edits', 'sum'), ref_chars=('ref_chars', 'sum'), lines=('id', 'size'),
               language=('language', 'first'))
          .assign(CER=lambda d: d['edits'] / d['ref_chars']).reset_index())
wide = by_doc.pivot_table(index=['doc_id', 'language', 'lines'], columns='model', values='CER', observed=True)
wide = wide.reset_index().sort_values(best)
wide.to_csv(os.path.join(EVAL_DIR, 'cer_by_document.csv'), index=False)

fig, ax = plt.subplots(figsize=(8, 0.38 * len(wide) + 1.8))
y = np.arange(len(wide))
ax.hlines(y, wide[first], wide[best], color=GRID, linewidth=2)
for name in (first, best):
    ax.scatter(wide[name], y, s=60, color=COLORS[name], zorder=3, label=name, edgecolor='#fcfcfb', linewidth=1.5)
ax.vlines(wide[BASE], y - 0.35, y + 0.35, label='Pretrained baseline', **BASELINE_STYLE)
ax.set_yticks(y, [f'{d} · {l} · {n} lines' for d, l, n in zip(wide.doc_id, wide.language, wide.lines)], fontsize=8)
ax.invert_yaxis()
ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.set(xlabel='CER (lower is better)', xlim=(0, None))
ax.set_title('Test CER per document', pad=24)
ax.grid(axis='y', visible=False)
ax.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=3, fontsize=9)
save(fig, '07_test_cer_by_document')
show(wide, list(MODELS))

## Examples

Test lines where the augmented model (the best of these three runs) improved most and where it got worse, compared with the 5-epoch run.

In [ ]:
a = preds[(preds.model == best) & (preds.split == 'test')].set_index('id')
b = preds[(preds.model == first) & (preds.split == 'test')].set_index('id').loc[a.index]
cmp = pd.DataFrame({'reference': a['reference'], f'{first}': b['prediction'], f'{best}': a['prediction'],
                    'CER change': a['line_cer'] - b['line_cer']})
pd.set_option('display.max_colwidth', 80)
print('Most improved')
show(cmp.sort_values('CER change').head(8), ['CER change'], signed=True)
print('Got worse')
show(cmp.sort_values('CER change', ascending=False).head(8), ['CER change'], signed=True)
print(f'Lines better: {(cmp["CER change"] < 0).sum()}, worse: {(cmp["CER change"] > 0).sum()}, '
      f'unchanged: {(cmp["CER change"] == 0).sum()}')